# SmolLM2-360M-Instruct — INT8 LoRA fine-tuning

Same sequence as your original notebook: system check → dataset → preview → statistics → model/tokenizer checks → **token length summary → choose MAX_LENGTH** → tokenize → INT8 LoRA → training → save. Run top to bottom.


In [1]:
%pip install --upgrade peft

Looking in indexes: https://artifacts.dell.com/artifactory/api/pypi/python/simple, https://artifacts.dell.com/artifactory/api/pypi/ailfc-1003745-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aia-1001238-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aiops-1002685-pypi-prd-local/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 368.1 MB/s  0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.12.0
    Uninstalling peft-0.12.0:
      Successfully uninstalled peft-0.12.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Check installed libraries (do not downgrade bitsandbytes 0.50.2)
import torch
import bitsandbytes as bnb
import transformers, peft

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("BitsAndBytes:", bnb.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
assert torch.cuda.is_available(), "CUDA GPU required"
print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.5.1+cu124
CUDA: 12.4
BitsAndBytes: 0.50.2
Transformers: 4.46.3
PEFT: 0.21.3
GPU: NVIDIA H100 80GB HBM3 MIG 2g.20gb


In [2]:
# import peft
# import bitsandbytes as bnb
# import torch

# print("PEFT:", peft.__version__)
# print("BitsAndBytes:", bnb.__version__)
# print("PyTorch:", torch.__version__)

In [3]:
# Load the same training file as the original notebook
import json

train_file = "/home/dell/SLM_Engineering/V1_Training_Data_Battery_Not_Charging_2000.jsonl"
training_data = []
with open(train_file, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, 1):
        if line.strip():
            training_data.append(json.loads(line))

assert training_data, "No training examples found"
assert all("messages" in example for example in training_data), "Expected messages-format JSONL"
print("Loaded examples:", len(training_data))
print("First example roles:", [m["role"] for m in training_data[0]["messages"]])


Loaded examples: 2000
First example roles: ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant']


In [4]:
print("=" * 70)
print("       FIRST 5 TRAINING EXAMPLES — PREVIEW")
print("=" * 70)

for i, example in enumerate(training_data[:5]):
    print(f"\n{'─' * 70}")
    print(f"📝 EXAMPLE {i+1}")
    print(f"{'─' * 70}")

    # Handle both formats
    if "messages" in example:
        system_text = ""
        user_text = ""
        assistant_text = ""
        for msg in example["messages"]:
            if msg["role"] == "system":
                system_text = msg["content"]
            elif msg["role"] == "user":
                user_text = msg["content"]
            elif msg["role"] == "assistant":
                assistant_text = msg["content"]
    else:
        system_text = example.get("system", "")
        user_text = example.get("user", "")
        assistant_text = example.get("assistant", "")

    # Show system prompt (truncated — it's the same for all)
    print(f"\n🔧 SYSTEM: {system_text[:80]}...")
    print(f"   (length: {len(system_text)} chars)")

    # Show user input
    print(f"\n👤 USER:")
    # Extract device_id from user text
    for line in user_text.split("\n"):
        if "device_id" in line:
            print(f"   {line.strip()}")
        if "calibrated_confidence" in line:
            print(f"   {line.strip()}")
        if "evidence_tier" in line:
            print(f"   {line.strip()}")

    # Show first SHAP feature
    shap_lines = [l.strip() for l in user_text.split("\n") if l.strip().startswith("- ")]
    if shap_lines:
        print(f"   Top SHAP: {shap_lines[0]}")
        print(f"   ({len(shap_lines)} features total)")

    # Show assistant response summary
    print(f"\n🤖 ASSISTANT:")
    print(f"   (length: {len(assistant_text)} chars)")

    # Extract key outputs
    for line in assistant_text.split("\n"):
        line_stripped = line.strip()
        if line_stripped.startswith("L3:"):
            print(f"   ✅ {line_stripped}")
        if line_stripped.startswith("Confidence:"):
            print(f"   📊 {line_stripped}")
        if line_stripped.startswith("- Function:"):
            print(f"   💊 {line_stripped}")

print(f"\n{'=' * 70}")
print(f"📊 Total examples: {len(training_data)}")
print(f"{'=' * 70}")

       FIRST 5 TRAINING EXAMPLES — PREVIEW

──────────────────────────────────────────────────────────────────────
📝 EXAMPLE 1
──────────────────────────────────────────────────────────────────────

🔧 SYSTEM: You are the Reasoning & Execution engine for Battery Not Charging issues.

Your ...
   (length: 2163 chars)

👤 USER:

🤖 ASSISTANT:
   (length: 827 chars)

──────────────────────────────────────────────────────────────────────
📝 EXAMPLE 2
──────────────────────────────────────────────────────────────────────

🔧 SYSTEM: You are the Reasoning & Execution engine for Battery Not Charging issues.

Your ...
   (length: 2163 chars)

👤 USER:

🤖 ASSISTANT:
   (length: 980 chars)

──────────────────────────────────────────────────────────────────────
📝 EXAMPLE 3
──────────────────────────────────────────────────────────────────────

🔧 SYSTEM: You are the Reasoning & Execution engine for Battery Not Charging issues.

Your ...
   (length: 2163 chars)

👤 USER:

🤖 ASSISTANT:
   (length: 970 char

In [5]:
print("=" * 70)
print("       DATASET STATISTICS")
print("=" * 70)

# Collect stats
l3_counts = {}
confidence_counts = {}
remediation_counts = {}
system_lengths = []
user_lengths = []
assistant_lengths = []

for example in training_data:
    # Handle both formats
    if "messages" in example:
        system_text = ""
        user_text = ""
        assistant_text = ""
        for msg in example["messages"]:
            if msg["role"] == "system": system_text = msg["content"]
            elif msg["role"] == "user": user_text = msg["content"]
            elif msg["role"] == "assistant": assistant_text = msg["content"]
    else:
        system_text = example.get("system", "")
        user_text = example.get("user", "")
        assistant_text = example.get("assistant", "")

    system_lengths.append(len(system_text))
    user_lengths.append(len(user_text))
    assistant_lengths.append(len(assistant_text))

    # Extract L3
    for line in assistant_text.split("\n"):
        line_s = line.strip()
        if line_s.startswith("L3:") and "Primary" not in line_s and "Secondary" not in line_s:
            l3 = line_s.replace("L3:", "").strip()
            l3_counts[l3] = l3_counts.get(l3, 0) + 1
        elif line_s.startswith("L3 (Primary):"):
            l3 = line_s.replace("L3 (Primary):", "").strip()
            l3_counts[l3] = l3_counts.get(l3, 0) + 1
        if line_s.startswith("Confidence:"):
            conf = line_s.replace("Confidence:", "").strip()
            confidence_counts[conf] = confidence_counts.get(conf, 0) + 1
        if line_s.startswith("- Function:"):
            func = line_s.replace("- Function:", "").strip()
            remediation_counts[func] = remediation_counts.get(func, 0) + 1

# Print L3 distribution
print("\n📊 L3 Root Cause Distribution:")
print("-" * 40)
for l3, count in sorted(l3_counts.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f"  {l3[:40]:<42} {count:>2} {bar}")

# Print confidence distribution
print(f"\n📊 Confidence Distribution:")
print("-" * 40)
for conf, count in sorted(confidence_counts.items()):
    print(f"  {conf:<15} {count:>2}")

# Print remediation distribution
print(f"\n📊 Remediation Functions Used:")
print("-" * 40)
for func, count in sorted(remediation_counts.items(), key=lambda x: -x[1]):
    print(f"  {func:<35} {count:>2}")

# Print length stats
print(f"\n📏 Text Lengths (characters):")
print("-" * 40)
print(f"  System prompt: {min(system_lengths):,} - {max(system_lengths):,} (same for all: {'YES' if len(set(system_lengths))==1 else 'NO'})")
print(f"  User input:    {min(user_lengths):,} - {max(user_lengths):,} (avg: {sum(user_lengths)//len(user_lengths):,})")
print(f"  Assistant:     {min(assistant_lengths):,} - {max(assistant_lengths):,} (avg: {sum(assistant_lengths)//len(assistant_lengths):,})")

print(f"\n{'=' * 70}")
print(f"✅ Dataset is ready for fine-tuning!")
print(f"   Total examples: {len(training_data)}")
print(f"{'=' * 70}")

       DATASET STATISTICS

📊 L3 Root Cause Distribution:
----------------------------------------

📊 Confidence Distribution:
----------------------------------------

📊 Remediation Functions Used:
----------------------------------------

📏 Text Lengths (characters):
----------------------------------------
  System prompt: 2,163 - 2,163 (same for all: YES)
  User input:    68 - 108 (avg: 100)
  Assistant:     647 - 1,072 (avg: 893)

✅ Dataset is ready for fine-tuning!
   Total examples: 2000


In [6]:
# Verify the original local model path
from pathlib import Path
from transformers import AutoConfig

model_dir = "/home/dell/SLM_Engineering/SLM_Models/SmolLM2-360M-Instruct"
assert Path(model_dir).is_dir(), f"Model directory not found: {model_dir}"
config = AutoConfig.from_pretrained(model_dir, local_files_only=True)
print("Model type:", config.model_type)
print("Model directory:", model_dir)


Model type: llama
Model directory: /home/dell/SLM_Engineering/SLM_Models/SmolLM2-360M-Instruct


In [7]:

from transformers import AutoTokenizer

model_dir = "/home/dell/SLM_Engineering/SLM_Models/SmolLM2-360M-Instruct"

# ============================================================
# Load tokenizer
# ============================================================
print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token
print(f"✅ Tokenizer loaded!")
print(f"   Vocab size: {tokenizer.vocab_size}")
print(f"   Pad token: {tokenizer.pad_token}")

# ============================================================
# Measure exact token length for EVERY example
# ============================================================
print(f"\n{'=' * 70}")
print(f"       TOKEN LENGTH ANALYSIS — ALL {len(training_data)} EXAMPLES")
print(f"{'=' * 70}")

token_lengths = []

for i, example in enumerate(training_data):
    # Handle both formats
    if "messages" in example:
        messages = example["messages"]
    else:
        # Convert flat format to messages
        messages = [
            {"role": "system", "content": example.get("system", "")},
            {"role": "user", "content": example.get("user", "")},
            {"role": "assistant", "content": example.get("assistant", "")}
        ]

    # Apply chat template (same as training will do)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Count tokens
    tokens = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    num_tokens = tokens["input_ids"].shape[1]
    token_lengths.append(num_tokens)

    # Extract device_id for display
    user_content = messages[1]["content"] if len(messages) > 1 else ""
    device_id = "unknown"
    for line in user_content.split("\n"):
        if "device_id" in line:
            device_id = line.split(":")[-1].strip()
            break

    # Color-code by length
    if num_tokens > 2560:
        status = "🔴"
    elif num_tokens > 2048:
        status = "🟡"
    else:
        status = "🟢"

    bar = "█" * (num_tokens // 50)
    if i < 10:
        print(f"  {status} Example {i+1:>2} ({device_id:>8}): {num_tokens:>5} tokens")

# ============================================================
# Summary statistics
# ============================================================
print(f"\n{'=' * 70}")
print(f"📊 TOKEN LENGTH SUMMARY")
print(f"{'=' * 70}")
print(f"   Minimum:  {min(token_lengths):>5} tokens")
print(f"   Maximum:  {max(token_lengths):>5} tokens")
print(f"   Average:  {sum(token_lengths) // len(token_lengths):>5} tokens")
print(f"   Median:   {sorted(token_lengths)[len(token_lengths)//2]:>5} tokens")

# ============================================================
# Recommend max_length
# ============================================================
max_tokens = max(token_lengths)

if max_tokens <= 1536:
    recommended_max = 1536
elif max_tokens <= 2048:
    recommended_max = 2048
elif max_tokens <= 2560:
    recommended_max = 2560
elif max_tokens <= 3072:
    recommended_max = 3072
else:
    recommended_max = 4096

# Count how many fit at each threshold
for threshold in [1536, 2048, 2560, 3072, 4096]:
    fits = sum(1 for t in token_lengths if t <= threshold)
    pct = 100 * fits / len(token_lengths)
    marker = " ✅ RECOMMENDED" if threshold == recommended_max else ""
    print(f"   max_length={threshold}: {fits}/{len(token_lengths)} fit ({pct:.0f}%){marker}")

print(f"\n🎯 Recommended max_length = {recommended_max}")
print(f"   (measured from the actual training examples)")

# Check for truncation risk
truncated = sum(1 for t in token_lengths if t > recommended_max)
if truncated > 0:
    print(f"\n⚠️  WARNING: {truncated} examples will be truncated!")
else:
    print(f"\n✅ ZERO examples will be truncated — perfect!")

⏳ Loading tokenizer...
✅ Tokenizer loaded!
   Vocab size: 49152
   Pad token: <|im_end|>

       TOKEN LENGTH ANALYSIS — ALL 2000 EXAMPLES
  🟢 Example  1 ( unknown):  1280 tokens
  🟢 Example  2 ( unknown):  1425 tokens
  🟢 Example  3 ( unknown):  1410 tokens
  🟢 Example  4 ( unknown):  1474 tokens
  🟢 Example  5 ( unknown):  1107 tokens
  🟢 Example  6 ( unknown):  1007 tokens
  🟢 Example  7 ( unknown):  1254 tokens
  🟢 Example  8 ( unknown):  1424 tokens
  🟢 Example  9 ( unknown):  1024 tokens
  🟢 Example 10 ( unknown):  1405 tokens

📊 TOKEN LENGTH SUMMARY
   Minimum:    992 tokens
   Maximum:   1530 tokens
   Average:   1303 tokens
   Median:    1290 tokens
   max_length=1536: 2000/2000 fit (100%) ✅ RECOMMENDED
   max_length=2048: 2000/2000 fit (100%)
   max_length=2560: 2000/2000 fit (100%)
   max_length=3072: 2000/2000 fit (100%)
   max_length=4096: 2000/2000 fit (100%)

🎯 Recommended max_length = 1536
   (measured from the actual training examples)

✅ ZERO examples will be truncate

### Choose sequence length after seeing the token summary

The next cell uses the recommended value calculated above. It is **not** set in the first cell. You can change it here if you deliberately accept truncation.


In [8]:
# Choose MAX_LENGTH only after measuring the dataset
MAX_LENGTH = 1600

# SmolLM2 supports a finite context window; do not silently exceed it.
context_limit = getattr(config, "max_position_embeddings", None)
if context_limit is not None and MAX_LENGTH > context_limit:
    raise ValueError(f"MAX_LENGTH={MAX_LENGTH} exceeds model context={context_limit}")

print("Selected MAX_LENGTH:", MAX_LENGTH)
print("Examples exceeding this length:", sum(n > MAX_LENGTH for n in token_lengths))


Selected MAX_LENGTH: 1600
Examples exceeding this length: 0


In [9]:
# Tokenize the original multi-turn messages (same chat template)
from datasets import Dataset

# Fixed-length padding, like the original notebook.
# Labels for padding must be -100 so padding does not affect the loss.
def format_and_tokenize(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    encoded = tokenizer(
        text, add_special_tokens=False,
        truncation=True, max_length=MAX_LENGTH, padding="max_length"
    )
    encoded["labels"] = [
        token if mask == 1 else -100
        for token, mask in zip(encoded["input_ids"], encoded["attention_mask"])
    ]
    return encoded

tokenized_dataset = Dataset.from_list([
    format_and_tokenize(example) for example in training_data
])
print("Tokenized examples:", len(tokenized_dataset))
print("Sequence length:", len(tokenized_dataset[0]["input_ids"]))
print("Ignored padding labels:", tokenized_dataset[0]["labels"].count(-100))


Tokenized examples: 2000
Sequence length: 1600
Ignored padding labels: 320


In [10]:
# Load the INT8 base model and apply the SAME LoRA settings
import torch
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False
)

model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    local_files_only=True
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

int8_layers = sum(isinstance(m, bnb.nn.Linear8bitLt) for m in model.modules())
assert int8_layers > 0, "No INT8 layers found"
print("INT8 layers:", int8_layers)
model.print_trainable_parameters()


INT8 layers: 224
trainable params: 8,683,520 || all params: 370,504,640 || trainable%: 2.3437


In [12]:
# Check the ACTUAL model before starting full training
# This does not update the LoRA weights.

import logging
import warnings

# Suppress MatMul8bitLt warnings
warnings.filterwarnings(
    "ignore",
    message=r".*MatMul8bitLt.*",
    category=UserWarning
)

# Suppress bitsandbytes warning logs
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

import torch

sample = tokenized_dataset[0]
inputs = {
    key: torch.tensor([sample[key]], device="cuda", dtype=torch.long)
    for key in ["input_ids", "attention_mask", "labels"]
}

model.train()
model.zero_grad(set_to_none=True)
try:
    with torch.autocast("cuda", dtype=torch.bfloat16):
        loss = model(**inputs).loss
    torch.cuda.synchronize()
    print("Forward pass OK. Loss:", loss.item())
    loss.backward()
    torch.cuda.synchronize()
    print("Backward pass OK")
finally:
    model.zero_grad(set_to_none=True)
    del inputs
    torch.cuda.empty_cache()


Forward pass OK. Loss: 2.016822338104248
Backward pass OK


In [14]:
# Train using the original hyperparameters
from transformers import TrainingArguments, Trainer, default_data_collator
import math

output_dir = "/home/dell/SLM_Engineering/V1_Output_Battery_Not Charging"
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    max_grad_norm=0.3,
    weight_decay=0.01,
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=3,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=default_data_collator
)

updates_per_epoch = math.ceil(len(tokenized_dataset) / 4)
print("Examples:", len(tokenized_dataset))
print("Epochs:", training_args.num_train_epochs)
print("Optimizer updates per epoch (approx):", updates_per_epoch)
print("Total optimizer updates (approx):", updates_per_epoch * 3)

training_completed = False
train_result = trainer.train()
training_completed = True
print("Training complete. Loss:", train_result.metrics["train_loss"])


Examples: 2000
Epochs: 3
Optimizer updates per epoch (approx): 500
Total optimizer updates (approx): 1500


Step,Training Loss
1,2.271300
2,2.081800
3,2.127600
4,2.142200
5,2.172200
6,2.010600
7,1.935600
8,1.857300
9,1.777100
10,1.988600


Training complete. Loss: 0.05257999427864949


In [ ]:
# Save only after successful training
import os

if not globals().get("training_completed", False):
    raise RuntimeError("Training did not finish; adapters will not be saved")

adapter_dir = "/home/dell/SLM_Engineering/V1_Final_INT8_2000_adapter_smollm"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Saved LoRA adapter to:", adapter_dir)
print("Files:", sorted(os.listdir(adapter_dir)))
